Dataset extraction from kaggle

In [ ]:
# Install Kaggle API
!pip install -q kaggle

# Upload kaggle.json (API key)
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"aishwaryapragada","key":"71f1d01fe50059c24c94df7d76cce6e4"}'}

In [ ]:
import os

# Create Kaggle directory
os.makedirs('/root/.kaggle', exist_ok=True)

# Move kaggle.json to correct location
!cp kaggle.json /root/.kaggle/

# Set proper permissions
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
# ==============================
# 📂 Download & Extract Dataset (Smart Way)
# ==============================

import os
import zipfile

dataset_path = "dataset"
zip_file = "sentinel-1-sar-oil-spill-detection-dataset.zip"

# Check if dataset already exists
if not os.path.exists(dataset_path):
    print("📥 Downloading dataset...")
    !kaggle datasets download -d harikrishnacs/sentinel-1-sar-oil-spill-detection-dataset

    print("📦 Extracting dataset...")
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(dataset_path)

    print("✅ Dataset downloaded and ready!")
else:
    print("✅ Dataset already exists. Skipping download.")

📥 Downloading dataset...
Dataset URL: https://www.kaggle.com/datasets/harikrishnacs/sentinel-1-sar-oil-spill-detection-dataset
License(s): CC-BY-SA-4.0
100% 65.3M/65.3M [00:01<00:00, 65.1MB/s]

📦 Extracting dataset...
✅ Dataset downloaded and ready!


In [ ]:
import os

print("📂 Dataset Structure Preview:\n")

for root, dirs, files in os.walk('dataset'):
    print(f"📁 {root}")
    print(f"   📁 Subfolders: {len(dirs)}")
    print(f"   📄 Files: {len(files)}")

    # Show only first few examples
    if dirs:
        print("   Example folders:", dirs[:3])
    if files:
        print("   Example files:", files[:5])

    print("-" * 40)

📂 Dataset Structure Preview:

📁 dataset
   📁 Subfolders: 2
   📄 Files: 0
   Example folders: ['metadata', 'data']
----------------------------------------
📁 dataset/metadata
   📁 Subfolders: 0
   📄 Files: 5
   Example files: ['readme-help.md', 'dublincore-000057430v001.xml', 'dublincore-collection.xml', 'collection_import_sha256sum.txt', 'Creative Commons Attribution-Share Alike.html']
----------------------------------------
📁 dataset/data
   📁 Subfolders: 3
   📄 Files: 0
   Example folders: ['S1SAR_UnBalanced_400by400_Class_1', 'S1SAR_UnBalanced_400by400_Class_0', 'Samples']
----------------------------------------
📁 dataset/data/S1SAR_UnBalanced_400by400_Class_1
   📁 Subfolders: 1
   📄 Files: 0
   Example folders: ['1']
----------------------------------------
📁 dataset/data/S1SAR_UnBalanced_400by400_Class_1/1
   📁 Subfolders: 0
   📄 Files: 1905
   Example files: ['10_400_400_img_yWdUTGXnMxebGon0_GIB_cls_1.jpg', '3_600_0_img_HZqm2V0YPVLx2ixE_JAV_cls_1.jpg', '14_400_600_img_fsFvbCAnE

Load Dataset

In [ ]:
import cv2
import numpy as np
import os
from tqdm import tqdm

IMG_SIZE = 128

class_0_path = 'dataset/data/S1SAR_UnBalanced_400by400_Class_0/0'
class_1_path = 'dataset/data/S1SAR_UnBalanced_400by400_Class_1/1'

def load_images(folder, label):
    images, labels = [], []

    for file in tqdm(os.listdir(folder)):
        if not file.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
            continue

        path = os.path.join(folder, file)

        try:
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = img / 255.0

            images.append(img)
            labels.append(label)

        except:
            continue

    return images, labels

print("📥 Loading Class 0...")
data_0, labels_0 = load_images(class_0_path, 0)

print("📥 Loading Class 1...")
data_1, labels_1 = load_images(class_1_path, 1)

X = np.array(data_0 + data_1)
y = np.array(labels_0 + labels_1)

print("✅ Dataset Loaded")
print("Shape:", X.shape)

📥 Loading Class 0...


100%|██████████| 3725/3725 [00:03<00:00, 979.88it/s]


📥 Loading Class 1...


100%|██████████| 1905/1905 [00:02<00:00, 658.77it/s]


✅ Dataset Loaded
Shape: (5630, 128, 128)


Class Distribution

In [ ]:
import numpy as np

unique, counts = np.unique(y, return_counts=True)

print("📊 Class Distribution:")
for cls, count in zip(unique, counts):
    print(f"Class {cls}: {count} samples")

📊 Class Distribution:
Class 0: 3725 samples
Class 1: 1905 samples


Feature Extraction

In [ ]:
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from skimage.measure import shannon_entropy
from scipy.stats import skew, kurtosis
from tqdm import tqdm
import numpy as np

def extract_features(X):
    feature_list = []

    for img in tqdm(X):
        img_uint8 = (img * 255).astype(np.uint8)
        features = []

        # -------- GLCM  --------
        glcm = graycomatrix(
            img_uint8,
            distances=[1, 2, 3],
            angles=[0, np.pi/4, np.pi/2],
            levels=256,
            symmetric=True,
            normed=True
        )

        props = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']
        for prop in props:
            features.append(graycoprops(glcm, prop).mean())

        # -------- LBP --------
        lbp = local_binary_pattern(img_uint8, P=8, R=1, method='uniform')

        hist, _ = np.histogram(
            lbp.ravel(),
            bins=np.arange(0, 11),
            range=(0, 10)
        )

        hist = hist.astype("float")
        hist /= (hist.sum() + 1e-6)
        features.extend(hist)

        # -------- Statistical --------
        features.append(np.mean(img))
        features.append(np.std(img))
        features.append(np.var(img))

        # -------- Entropy --------
        features.append(shannon_entropy(img))

        # -------- Stable Skew/Kurtosis --------
        flat = img.flatten()
        if np.std(flat) > 1e-6:
            features.append(skew(flat))
            features.append(kurtosis(flat))
        else:
            features.append(0)
            features.append(0)

        feature_list.append(features)

    return np.array(feature_list)

# Extract features
X_features = extract_features(X)

print("✅ Feature extraction done")
print("Shape:", X_features.shape)

100%|██████████| 5630/5630 [07:53<00:00, 11.89it/s]

✅ Feature extraction done
Shape: (5630, 21)


Saving features

In [ ]:
import os
import numpy as np

os.makedirs("saved_data", exist_ok=True)

np.save("saved_data/features.npy", X_features)
np.save("saved_data/labels.npy", y)

print("✅ Features and labels saved successfully!")

✅ Features and labels saved successfully!


Train-Test Split + Scaling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scaling (NEW IMPROVEMENT)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("✅ Data split and scaling done")

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\n📊 Train distribution:", np.bincount(y_train))
print("📊 Test distribution:", np.bincount(y_test))

✅ Data split and scaling done
Train shape: (4504, 21)
Test shape: (1126, 21)

📊 Train distribution: [2980 1524]
📊 Test distribution: [745 381]


SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE
import numpy as np

# Apply SMOTE with tuned parameter
smote = SMOTE(k_neighbors=3, random_state=42)

X_train, y_train = smote.fit_resample(X_train, y_train)

print("✅ SMOTE applied successfully")

# Check new distribution
unique, counts = np.unique(y_train, return_counts=True)

print("\n📊 After SMOTE:")
for cls, count in zip(unique, counts):
    print(f"Class {cls}: {count} samples")

✅ SMOTE applied successfully

📊 After SMOTE:
Class 0: 2980 samples
Class 1: 2980 samples


Hyperparameter Tuning (GridSearch)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Parameter grid
param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [15, 20, 25],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    verbose=2,
    n_jobs=-1
)

# Train
grid_search.fit(X_train, y_train)

print("\n✅ Best Parameters:", grid_search.best_params_)
print("Best Cross-validation Accuracy:", grid_search.best_score_)

# Final model
rf_model = grid_search.best_estimator_

Fitting 3 folds for each of 24 candidates, totalling 72 fits

✅ Best Parameters: {'max_depth': 25, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Best Cross-validation Accuracy: 0.9412776704166204


In [ ]:
# ==============================
# 📊 Final Model Evaluation
# ==============================

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# Predictions
y_pred = rf_model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

# ROC-AUC
y_prob = rf_model.predict_proba(X_test)[:, 1]
roc = roc_auc_score(y_test, y_prob)

print("✅ Model Evaluation Completed\n")
print(f"Accuracy: {accuracy:.4f}")
print(f"ROC-AUC Score: {roc:.4f}")

print("\n📌 Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\n📌 Classification Report:")
print(classification_report(y_test, y_pred))

✅ Model Evaluation Completed

Accuracy: 0.9405
ROC-AUC Score: 0.9813

📌 Confusion Matrix:
[[710  35]
 [ 32 349]]

📌 Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.95      0.95       745
           1       0.91      0.92      0.91       381

    accuracy                           0.94      1126
   macro avg       0.93      0.93      0.93      1126
weighted avg       0.94      0.94      0.94      1126



In [ ]:
# ==============================
# 💾 Save Model + Scaler
# ==============================

import joblib
import os

os.makedirs("saved_models", exist_ok=True)

# Save model
joblib.dump(rf_model, "saved_models/rf_model.pkl")

# Save scaler
joblib.dump(scaler, "saved_models/scaler.pkl")

print("✅ Model and scaler saved successfully!")

✅ Model and scaler saved successfully!


In [ ]:
!pip install -q streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 74.2 MB/s eta 0:00:00


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3BLnBMYeMUqeCCPJYB5yCSUKcPo_cGFmXEc7bsXb38D88WGs")

In [ ]:
%%writefile app.py
import streamlit as st
st.set_page_config(page_title="Oil Spill Detection", layout="centered")

# ===== CLEAN OCEAN UI =====
st.markdown("""
<style>

/* Remove top blank space */
.block-container {
    padding-top: 0rem !important;
    padding-bottom: 2rem;
}

/* Deep ocean background */
.stApp {
    background: linear-gradient(180deg, #011627, #023E5A, #03506F);
}

/* Centered card */
.custom-card {
    background: rgba(255, 255, 255, 0.03);
    padding: 30px;
    border-radius: 10px;
    max-width: 850px;
    margin: 40px auto;
    box-shadow: 0px 6px 30px rgba(0,0,0,0.7);
}

/* Title */
h1 {
    text-align: center;
    color: #E6F3FF;
    font-weight: 600;
}

/* Subtitle */
p {
    text-align: center;
    color: #A9D6E5;
    font-size: 15px;
}

/* Button styling */
.stButton>button {
    width: 100%;
    border-radius: 6px;
    background: #0288D1;
    color: white;
    font-weight: 500;
    padding: 10px;
    border: none;
}

/* Image headings */
h3 {
    color: #CFE8F3;
    text-align: center;
}

.result-box {
    padding: 15px;
    border-radius: 8px;
    text-align: center;
    font-size: 18px;
    font-weight: 600;
    margin-top: 10px;
}

.detected {
    background-color: rgba(255, 0, 0, 0.2);
    color: #ff4d4d;
    border: 1px solid #ff4d4d;
}

.not-detected {
    background-color: rgba(0, 255, 150, 0.15);
    color: #00e676;
    border: 1px solid #00e676;
}
</style>
""", unsafe_allow_html=True)

# ===== IMPORTS =====
import cv2
import numpy as np
import joblib
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from scipy.stats import skew, kurtosis
from skimage.measure import shannon_entropy

# ===== LOAD MODEL + SCALER (FIXED) =====
model = joblib.load("saved_models/rf_model.pkl")
scaler = joblib.load("saved_models/scaler.pkl")

IMG_SIZE = 128

# ===== FEATURE EXTRACTION (MATCHES TRAINING) =====
def extract_features(img):
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = gray / 255.0
    img_uint8 = (gray * 255).astype(np.uint8)

    features = []

    # ✅ Correct GLCM
    glcm = graycomatrix(
        img_uint8,
        distances=[1, 2, 3],
        angles=[0, np.pi/4, np.pi/2],
        levels=256,
        symmetric=True,
        normed=True
    )

    props = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']
    for prop in props:
        features.append(graycoprops(glcm, prop).mean())

    # ✅ LBP
    lbp = local_binary_pattern(img_uint8, P=8, R=1, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 11), range=(0, 10))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)
    features.extend(hist)

    # ✅ Stats
    features.append(np.mean(gray))
    features.append(np.std(gray))
    features.append(np.var(gray))

    # ✅ Entropy
    features.append(shannon_entropy(gray))

    # ✅ Skew/Kurtosis safe
    flat = gray.flatten()
    if np.std(flat) > 1e-6:
        features.append(skew(flat))
        features.append(kurtosis(flat))
    else:
        features.append(0)
        features.append(0)

    return np.array(features).reshape(1, -1)

# ===== OIL SPILL HIGHLIGHT =====
def highlight_oil_spill(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)

    _, thresh = cv2.threshold(blur, 60, 255, cv2.THRESH_BINARY_INV)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    output = image.copy()

    for cnt in contours:
        if cv2.contourArea(cnt) > 500:
            cv2.drawContours(output, [cnt], -1, (0, 0, 255), 2)

    return output

# ===== PREDICTION =====
def predict_image(image):
    features = extract_features(image)

    # ✅ IMPORTANT: scaling
    features = scaler.transform(features)

    prediction = model.predict(features)[0]

    if prediction == 1:
        marked_image = highlight_oil_spill(image)
        return "Oil Spill Detected", marked_image
    else:
        return "No Oil Spill", image

# ===== UI =====
st.markdown('<div class="custom-card">', unsafe_allow_html=True)

st.title("Oil Spill Detection System")
st.markdown("Upload a satellite image to detect oil spill regions.")
st.divider()

uploaded_file = st.file_uploader("Upload SAR Image", type=["jpg", "png", "jpeg"])

col1, col2 = st.columns(2)

if uploaded_file is not None:
    file_bytes = np.asarray(bytearray(uploaded_file.read()), dtype=np.uint8)
    image = cv2.imdecode(file_bytes, 1)

    with col1:
        st.subheader("Input Image")
        st.image(cv2.cvtColor(image, cv2.COLOR_BGR2RGB), use_container_width=True)

    if st.button("Detect Oil Spill"):
        with st.spinner("Detecting oil spill... Please wait"):
            result, output_img = predict_image(image)

        with col2:
            st.subheader("Detection Result")
            st.image(cv2.cvtColor(output_img, cv2.COLOR_BGR2RGB), use_container_width=True)

        st.divider()

        if "Detected" in result:
            st.markdown(f'<div class="result-box detected">{result}</div>', unsafe_allow_html=True)
        else:
            st.markdown(f'<div class="result-box not-detected">{result}</div>', unsafe_allow_html=True)

st.markdown('</div>', unsafe_allow_html=True)

Writing app.py


In [ ]:
from pyngrok import ngrok

# kill previous tunnels
ngrok.kill()

# run streamlit in background
get_ipython().system_raw('streamlit run app.py &')

# create public link
public_url = ngrok.connect(8501)

print("👉 OPEN THIS LINK:")
print(public_url)

👉 OPEN THIS LINK:
NgrokTunnel: "https://paramorphic-irreversibly-kenda.ngrok-free.dev" -> "http://localhost:8501"
